<a href="https://colab.research.google.com/github/drchadvidden/courseMaterials/blob/main/SupervisedLearning/Labs/SL_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Multiple linear regression





# Lab Instructions

Run each of the coding cells. For tutorial example cells, understand the commands and check that the outputs make sense. For exercise cells, write your own code where indicated to generate the correct output. Give text explanations where indicated.

### Submission:
Complete the following notebook in order. Once done, save the notebook and upload the resulting .html file to the Canvas course assignment.

### Rubric:
15 total points, 5 points to running tutorial example cells and saving outputs, 10 points for completing exercises.

### Deadline:
Tuesday at midnight after the lab is assigned.

# Tutorial: Multiple Linear Regression

In [ ]:
pip install ISLP

## Importing packages
We import our standard libraries at this top
level.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots


### New imports
Throughout this lab we will introduce new functions and libraries. However,
we will import them here to emphasize these are the new
code objects in this lab. Keeping imports near the top
of a notebook makes the code more readable, since scanning the first few
lines tells us what libraries are used.

In [ ]:
import statsmodels.api as sm


 We will provide relevant details about the
functions below as they are needed.

Besides importing whole modules, it is also possible
to import only a few items from a given module. This
will help keep the  *namespace* clean.
We will use a few specific objects from the `statsmodels` package
which we import here.

In [ ]:
from statsmodels.stats.outliers_influence \
     import variance_inflation_factor as VIF
from statsmodels.stats.anova import anova_lm


As one of the import statements above is quite a long line, we inserted a line break `\` to
ease readability.

We will also use some functions written for the labs in this book in the `ISLP`
package.

In [ ]:
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)


### Inspecting Objects and Namespaces
The
function  `dir()`
provides a list of
objects in a namespace.

In [ ]:
dir()


 This shows you everything that `Python` can find at the top level.
There are certain objects like `__builtins__` that contain references to built-in
functions like `print()`.

Every python object has its own notion of
namespace, also accessible with `dir()`. This will include
both the attributes of the object
as well as any methods associated with it. For instance, we see `'sum'` in the listing for an
array.

In [ ]:
A = np.array([3,5,11])
dir(A)


 This indicates that the object `A.sum` exists. In this case it is a method
that can be used to compute the sum of the array `A` as can be seen by typing `A.sum?`.

In [ ]:
A.sum()


## Multiple Linear Regression
In order to fit a multiple linear regression model using least squares, we again use
the `ModelSpec()`  transform to construct the required
model matrix and response. The arguments
to `ModelSpec()` can be quite general, but in this case
a list of column names suffice. We consider a fit here with
the two variables `lstat` and `age`.

In [ ]:
Boston = load_data("Boston")
Boston.columns

In [ ]:
X = MS(['lstat', 'age']).fit_transform(Boston)
y = Boston['medv']
model1 = sm.OLS(y, X)
results1 = model1.fit()
summarize(results1)

Notice how we have compacted the first line into a succinct expression describing the construction of `X`.

The  `Boston`   data set contains 12 variables, and so it would be cumbersome
to have to type all of these in order to perform a regression using all of the predictors.
Instead, we can use the following short-hand:\definelongblankMR{columns.drop()}{columns.slashslashdrop()}

In [ ]:
terms = Boston.columns.drop('medv')
terms


We can now fit the model with all the variables in `terms` using
the same model matrix builder.

In [ ]:
X = MS(terms).fit_transform(Boston)
model = sm.OLS(y, X)
results = model.fit()
summarize(results)


What if we would like to perform a regression using all of the variables but one?  For
example, in the above regression output,   `age`  has a high $p$-value.
So we may wish to run a regression excluding this predictor.
The following syntax results in a regression using all predictors except  `age`.

In [ ]:
minus_age = Boston.columns.drop(['medv', 'age'])
Xma = MS(minus_age).fit_transform(Boston)
model1 = sm.OLS(y, Xma)
summarize(model1.fit())


## Multivariate Goodness of Fit
We can access the individual components of `results` by name
(`dir(results)` shows us what is available). Hence
`results.rsquared` gives us the $R^2$,
and
`np.sqrt(results.scale)` gives us the RSE.

Variance inflation factors (section~\ref{Ch3:problems.sec}) are sometimes useful
to assess the effect of collinearity in the model matrix of a regression model.
We will compute the VIFs in our multiple regression fit, and use the opportunity to introduce the idea of *list comprehension*.

### List Comprehension
Often we encounter a sequence of objects which we would like to transform
for some other task. Below, we compute the VIF for each
feature in our `X` matrix and produce a data frame
whose index agrees with the columns of `X`.
The notion of list comprehension can often make such
a task easier.

List comprehensions are simple and powerful ways to form
lists of `Python` objects. The language also supports
dictionary and *generator* comprehension, though these are
beyond our scope here. Let's look at an example. We compute the VIF for each of the variables
in the model matrix `X`, using the function `variance_inflation_factor()`.


In [ ]:
vals = [VIF(X, i)
        for i in range(1, X.shape[1])]
vif = pd.DataFrame({'vif':vals},
                   index=X.columns[1:])
vif


The function `VIF()` takes two arguments: a dataframe or array,
and a variable column index. In the code above we call `VIF()` on the fly for all columns in `X`.  
We have excluded column 0 above (the intercept), which is not of interest. In this case the VIFs are not that exciting.

The object `vals` above could have been constructed with the following for loop:

In [ ]:
vals = []
for i in range(1, X.values.shape[1]):
    vals.append(VIF(X.values, i))


List comprehension allows us to perform such repetitive operations in a more straightforward way.
## Interaction Terms
It is easy to include interaction terms in a linear model using `ModelSpec()`.
Including a tuple `("lstat","age")` tells the model
matrix builder to include an interaction term between
 `lstat`  and  `age`.

In [ ]:
X = MS(['lstat',
        'age',
        ('lstat', 'age')]).fit_transform(Boston)
model2 = sm.OLS(y, X)
summarize(model2.fit())


## Non-linear Transformations of the Predictors
The model matrix builder can include terms beyond
just column names and interactions. For instance,
the `poly()` function supplied in `ISLP` specifies that
columns representing polynomial functions
of its first argument are added to the model matrix.

In [ ]:
X = MS([poly('lstat', degree=2), 'age']).fit_transform(Boston)
model3 = sm.OLS(y, X)
results3 = model3.fit()
summarize(results3)


The effectively zero *p*-value associated with the quadratic term
(i.e. the third row above) suggests that it leads to an improved model.

By default, `poly()` creates a basis matrix for inclusion in the
model matrix whose
columns are *orthogonal polynomials*, which are designed for stable
least squares computations. {Actually, `poly()` is a  wrapper for the workhorse and standalone  function `Poly()` that does the  work in building the model matrix.}
Alternatively, had we included an argument
`raw=True` in the above call to `poly()`, the basis matrix would consist simply of
`lstat` and `lstat**2`. Since either of these bases
represent quadratic polynomials, the fitted values  would not
change in this case, just the polynomial coefficients.  Also by default, the columns
created by `poly()` do not include an intercept column as
that is automatically added by `MS()`.

We use the `anova_lm()` function to further quantify the extent to which the quadratic fit is
superior to the linear fit.

In [ ]:
anova_lm(results1, results3)


Here `results1` represents the linear submodel containing
predictors `lstat` and `age`,
while `results3` corresponds to the larger model above  with a quadratic
term in `lstat`.
The `anova_lm()` function performs a hypothesis test
comparing the two models. The null hypothesis is that the quadratic
term in the bigger model is not needed, and the alternative hypothesis is that the
bigger model is superior. Here the *F*-statistic is 177.28 and
the associated *p*-value is zero.
In this case the *F*-statistic is the square of the
*t*-statistic for the quadratic term in the linear model summary
for `results3` --- a consequence of the fact that these nested
models differ by one degree of freedom.
This provides very clear evidence that the quadratic polynomial in
`lstat` improves the linear model.
This is not surprising, since earlier we saw evidence for non-linearity in the relationship between `medv`
and  `lstat`.

The function `anova_lm()` can take more than two nested models
as input, in which case it compares every successive pair of models.
That also explains why their are `NaN`s in the first row above, since
there is no previous model with which to compare the first.


In [ ]:
ax = subplots(figsize=(8,8))[1]
ax.scatter(results3.fittedvalues, results3.resid)
ax.set_xlabel('Fitted value')
ax.set_ylabel('Residual')
ax.axhline(0, c='k', ls='--');


We see that when the quadratic term is included in the model,
there is little discernible pattern in the residuals.
In order to create a cubic or higher-degree polynomial fit, we can simply change the degree argument
to `poly()`.


## Qualitative Predictors
Here we use the  `Carseats`  data, which is included in the
`ISLP` package. We will  attempt to predict `Sales`
(child car seat sales) in 400 locations based on a number of
predictors.

In [ ]:
Carseats = load_data('Carseats')
Carseats.columns


The `Carseats`  
 data includes qualitative predictors such as
 `ShelveLoc`, an indicator of the quality of the shelving
 location --- that is,
the  space within a store in which the car seat is displayed. The predictor
 `ShelveLoc`  takes on three possible values, `Bad`, `Medium`, and `Good`.
Given a qualitative variable such as  `ShelveLoc`, `ModelSpec()` generates dummy
variables automatically.
These variables are often referred to as a *one-hot encoding* of the categorical
feature. Their columns sum to one, so to avoid collinearity with an intercept, the first column is dropped. Below we see
the column `ShelveLoc[Bad]` has been dropped, since `Bad` is the first level of `ShelveLoc`.
Below we fit a multiple regression model that includes some interaction terms.

In [ ]:
allvars = list(Carseats.columns.drop('Sales'))
y = Carseats['Sales']
final = allvars + [('Income', 'Advertising'),
                   ('Price', 'Age')]
X = MS(final).fit_transform(Carseats)
model = sm.OLS(y, X)
summarize(model.fit())


In the first line above, we made `allvars` a list, so that we
could add the interaction terms two lines down.
Our model-matrix builder has created a `ShelveLoc[Good]`
dummy variable that takes on a value of 1 if the
shelving location is good, and 0 otherwise. It has also created a `ShelveLoc[Medium]`
dummy variable that equals 1 if the shelving location is medium, and 0 otherwise.
A bad shelving location corresponds to a zero for each of the two dummy variables.
The fact that the coefficient for `ShelveLoc[Good]` in the regression output is
positive indicates that a good shelving location is associated with high sales (relative to a bad location).
And `ShelveLoc[Medium]` has a smaller positive coefficient,
indicating that a medium shelving location leads to higher sales than a bad
shelving location, but lower sales than a good shelving location.



# Exercise(s):

---







## Exercise 1: Autos

This question involves the use of multiple linear regression on the Auto data set.

### Tasks:

1. Produce a scatterplot matrix which includes all of the variables in the data set. Compute the matrix of correlations between the variables using the `DataFrame.corr()` method. Comment on any notable relationships between the variables.

2. Use the `sm.OLS()` function to perform a multiple linear regression with `mpg` as the response and all other variables except `name` as the predictors. Use the `summarize()` function to print the results. Comment on the output. For example:

* Is there a relationship between the predictors and the response? Use the `anova_lm()` function from `statsmodels` to answer this question.
* Which predictors appear to have a statistically significant relationship to the response?
* What does the coefficient for the `year` variable suggest?

3. Produce some of the diagnostic plots of the linear regression fit as described in the lab. Comment on any problems you see with the fit. For example:

* Do the residual plots suggest any unusually large outliers?
* Does the leverage plot identify any observations with unusually high leverage?

4. Fit some models with interactions as described in the lab. Do any interactions appear to be statistically significant?

5. Try a few different transformations of the variables, such as $\log(X)$, $\sqrt{X}$, and $X^2$. Comment on your findings.


In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 2: Carseats

This question involves the use of multiple linear regression on the Carseats data set.

### Tasks:

1. Fit a multiple linear regression model to predict `Sales` using `Price`, `Urban`, and `US` as predictors. Use the `summarize()` function to print the results. Comment on the output.

2. Interpret each coefficient in the model. Be careful when interpreting the coefficients for the qualitative variables.

3. Write out the fitted model in equation form, being careful to handle the qualitative variables properly.

4. For which of the predictors can you reject the null hypothesis $H_0: \beta_j = 0$? Explain your reasoning.

5. Based on your response to the previous question, fit a smaller model that includes only the predictors for which there is evidence of an association with `Sales`.

6. Compare the models from Tasks 1 and 5. How well does each model fit the data?

7. Using the model from Task 5, obtain 95% confidence intervals for the coefficient(s). Interpret the intervals.

8. Produce diagnostic plots for the model from Task 5 as described in the lab. Comment on whether there is evidence of outliers or observations with unusually high leverage.


In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 3: Collinearity

This question focuses on the problem of collinearity in multiple linear regression.

### Tasks:

1. Generate the following data using Python:

```python
rng = np.random.default_rng(10)
x1 = rng.uniform(0, 1, size=100)
x2 = 0.5 * x1 + rng.normal(size=100) / 10
y = 2 + 2 * x1 + 0.3 * x2 + rng.normal(size=100)
```

The last line creates a linear model in which `y` is a function of `x1` and `x2`. Write out the form of the linear model. What are the true regression coefficients?

2. Compute the correlation between `x1` and `x2`. Create a scatterplot displaying the relationship between the two variables. Comment on the strength and direction of the relationship.

3. Using this data, fit a least squares regression model to predict `y` using `x1` and `x2`. Use the `summarize()` function to print the results. Comment on the output. For example:

* What are $\hat{\beta}_0$, $\hat{\beta}_1$, and $\hat{\beta}_2$?
* How do the estimated coefficients compare with the true coefficients?
* Can you reject the null hypothesis $H_0:\beta_1=0$?
* Can you reject the null hypothesis $H_0:\beta_2=0$?
* What does the output suggest about the effect of collinearity?

4. Fit a least squares regression model to predict `y` using only `x1`. Comment on the results. Can you reject the null hypothesis $H_0:\beta_1=0$?

5. Fit a least squares regression model to predict `y` using only `x2`. Comment on the results. Can you reject the null hypothesis $H_0:\beta_1=0$?

6. Do the results from Tasks 3–5 contradict each other? Explain why or why not. In particular, discuss how the correlation between `x1` and `x2` affects the interpretation and statistical significance of the regression coefficients.

7. Suppose we obtain one additional observation that was unfortunately mismeasured. Add the observation to each of `x1`, `x2`, and `y` using:

```python
x1 = np.concatenate([x1, [0.1]])
x2 = np.concatenate([x2, [0.8]])
y = np.concatenate([y, [6]])
```

Re-fit the linear regression models from Tasks 3–5 using the new data. Comment on the effect of this observation on each model. For each model:

* Is the observation an outlier?
* Is the observation a high-leverage point?
* Is it both?
* How does the observation affect the fitted regression coefficients?


In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




## Exercise 4: Boston Data

This question involves the use of simple and multiple linear regression on the Boston data set. The goal is to predict per capita crime rate using the other variables in the data set.

### Tasks:

1. For each predictor, fit a simple linear regression model to predict the per capita crime rate. Use the `summarize()` function to print the results. Describe your findings. For example:

* Which predictors have a statistically significant association with the response?
* What is the direction of each significant association?
* Create plots of the response versus the predictors to support your conclusions.

2. Fit a multiple linear regression model to predict the per capita crime rate using all of the predictors. Use the `summarize()` function to print the results. Describe your findings. For example:

* Which predictors have a statistically significant relationship with the response?
* How do the estimated coefficients compare with those from the simple regression models?

3. Compare the results from Tasks 1 and 2. Create a scatterplot with the simple linear regression coefficients from Task 1 on the x-axis and the multiple regression coefficients from Task 2 on the y-axis. Each point should represent one predictor. Comment on any notable differences between the two sets of coefficients. What might explain these differences?

4. Investigate whether there is evidence of non-linear relationships between the predictors and the response. For each predictor $X$, fit a polynomial regression model of the form

$$
Y = \beta_0 + \beta_1X + \beta_2X^2 + \beta_3X^3 + \epsilon.
$$

Use the regression results to determine whether the quadratic or cubic terms provide evidence of a non-linear association between the predictor and the response. Comment on your findings.


In [ ]:
# Write your code for the exercise here!

### Explain your findings here:




# HTML Export Code

In [ ]:
# code to export notebook as .html for Canvas upload

from google.colab import drive
from google.colab import files

drive.mount('/content/drive')

notebook_name = "SL_Lab_3"
!cp "/content/drive/MyDrive/Colab Notebooks/DSC 430/{notebook_name}.ipynb" /content/
!jupyter nbconvert --to html "/content/{notebook_name}.ipynb"
files.download(f"/content/{notebook_name}.html")

